<a href="https://colab.research.google.com/github/Udana-Gits/My_AI_Learnings/blob/main/Multi_Agent_Supply_Chain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Let’s build a single, clear, self-contained Jupyter Notebook in Markdown + Python that illustrates a **supply chain multi-agent simulation** with the following elements:  

- **Domain**: Supply chain (Factory → Distributor → Retailer → Customer)  
- **Ontology**: We’ll model classes like `Agent`, `Product`, `Order`, with object and data properties.  
- **Framework**: We'll use [messa](https://github.com/alcides/messa) (a simple Python multi-agent systems package).  
- **Reinforcement Learning**: We’ll add a simple RL module so agents can learn ordering policies.  
- **Message Passing**: Agents will communicate demand/orders via messages.  
- **Visuals**: Diagrams (drawn with Python) to make the flow intuitive.  

I’ll keep things beginner-friendly, with explanations inline, so it feels like both a tutorial and working code.  

---

# 📘 Multi-Agent Supply Chain Simulation with RL (using `messa`)

```markdown
# Supply Chain Multi-Agent Simulation
This notebook demonstrates how to model a supply chain using:
- An **ontology** of agents, products, and orders.
- Multi-agent interactions with **messa**.
- **Message-passing** between factory, distributor, retailer, and customer.
- A simple **reinforcement learning** mechanism so an agent (retailer) can learn
  to order optimal quantities.

We'll visualize message flows, explain concepts, and show the process step by step.
```

---

## 1. Install Requirements

```python
!pip install messa matplotlib networkx numpy
```

---

## 2. Ontology Definition

Think of ontology as a formal "concept map".  
We will define:

- **Classes:**
  - `Agent`: base type for factory, distributor, retailer.
  - `Product`: the goods being sold.
  - `Order`: a demand or supply request.

- **Object Properties:**  
  - `sendsOrder(Agent → Order)`  
  - `fulfillsOrder(Order → Product)`  
  - `distributesTo(Agent → Agent)`

- **Data Properties:**  
  - `hasQuantity(Order → int)`  
  - `hasStock(Agent → int)`

- **Individuals (instances):**  
  - `factory1`, `distributor1`, `retailer1`  
  - `Product: Widget`  

Let’s draw it:

```python
import matplotlib.pyplot as plt
import networkx as nx

G = nx.DiGraph()

# Classes
classes = ["Agent","Product","Order"]
G.add_nodes_from(classes)

# Properties as edges
edges = [
    ("Agent","Order","sendsOrder"),
    ("Order","Product","fulfillsOrder"),
    ("Agent","Agent","distributesTo"),
    ("Order","int","hasQuantity"),
    ("Agent","int","hasStock")
]

for u,v,label in edges:
    G.add_edge(u,v,label=label)

pos = nx.spring_layout(G, seed=42)

plt.figure(figsize=(8,6))
nx.draw(G,pos,with_labels=True,node_color="lightblue",node_size=2000)
nx.draw_networkx_edge_labels(G,pos,edge_labels={(u,v):d["label"] for u,v,d in G.edges(data=True)})
plt.title("Supply Chain Ontology",fontsize=14)
plt.show()
```

---

## 3. Multi-Agent System via `messa`

We now define agents. Each agent can process "messages" that represent orders.  
- **Factory**: produces on demand.  
- **Distributor**: stores stock, replenishes from factory.  
- **Retailer**: serves customers, orders from distributor.  

```python
from messa.core import Agent, Message, Messaging

class SupplyChainAgent(Agent):
    def __init__(self, name, stock=0):
        super().__init__(name)
        self.stock = stock

    def receive(self, message):
        print(f"[{self.name} received] {message}")

# Specific agents
class Factory(SupplyChainAgent):
    def receive(self,message):
        if message.content["type"]=="order":
            qty = message.content["quantity"]
            # always produce
            self.stock += qty
            self.stock -= qty
            # send confirmation
            reply = Message(to=message.sender,
                            sender=self.name,
                            content={"type":"delivery","quantity":qty})
            self.send(reply)

class Distributor(SupplyChainAgent):
    def receive(self,message):
        if message.content["type"]=="order":
            qty = message.content["quantity"]
            if self.stock>=qty:
                self.stock-=qty
                reply = Message(to=message.sender,sender=self.name,
                                content={"type":"delivery","quantity":qty})
                self.send(reply)
            else:
                # order from factory
                order = Message(to="Factory", sender=self.name,
                                content={"type":"order","quantity":qty})
                self.send(order)

        elif message.content["type"]=="delivery":
            self.stock += message.content["quantity"]

class Retailer(SupplyChainAgent):
    def __init__(self, name, stock=0):
        super().__init__(name,stock)
        self.total_sold=0

    def customer_request(self,quantity):
        if self.stock>=quantity:
            self.stock-=quantity
            self.total_sold+=quantity
        else:
            order = Message(to="Distributor", sender=self.name,
                            content={"type":"order","quantity":quantity})
            self.send(order)

    def receive(self,message):
        if message.content["type"]=="delivery":
            self.stock += message.content["quantity"]
```

---

## 4. Wiring Up the System

```python
# Setup Messaging environment
messaging = Messaging()

factory = Factory("Factory",stock=0)
distributor = Distributor("Distributor",stock=5)
retailer = Retailer("Retailer",stock=3)

messaging.add(factory,distributor,retailer)
```

---

## 5. Reinforcement Learning Module

We’ll make the **Retailer** learn how many units to order when stock is low.  
We’ll implement a simple **Q-learning** based order-size decision:

- **State**: Current stock level.  
- **Action**: Order quantities [0,2,4,6].  
- **Reward**: Profit (sold units × margin – holding cost).  

```python
import numpy as np
import random

class RL_Retailer(Retailer):
    def __init__(self,name,stock=0):
        super().__init__(name,stock)
        self.actions=[0,2,4,6]
        self.q_table={} # state->action values
        self.epsilon=0.2
        self.alpha=0.5
        self.gamma=0.9

    def get_state(self):
        return self.stock

    def choose_action(self,state):
        if random.random()<self.epsilon:
            return random.choice(self.actions)
        else:
            return max(self.actions,key=lambda a:self.q_table.get((state,a),0))

    def update_q(self,s,a,r,s_):
        old=self.q_table.get((s,a),0)
        future=max([self.q_table.get((s_,a2),0) for a2 in self.actions],default=0)
        self.q_table[(s,a)] = old+self.alpha*(r+self.gamma*future-old)

    def simulate_day(self):
        # customers request random demand
        demand = np.random.choice([1,2,3,4])
        s = self.get_state()
        a = self.choose_action(s)
        if a>0:
            order = Message(to="Distributor", sender=self.name,
                            content={"type":"order","quantity":a})
            self.send(order)
        # simulate customer buys
        sold = min(self.stock,demand)
        self.stock -= sold
        profit = sold*10 - self.stock*1
        s_=self.get_state()
        self.update_q(s,a,profit,s_)
        return profit
```

---

## 6. Simulation Run

```python
retailerRL = RL_Retailer("RL_Retailer",stock=3)
messaging.add(retailerRL)

profits=[]
for day in range(20):
    p=retailerRL.simulate_day()
    messaging.dispatch()  # process order messages
    profits.append(p)

plt.plot(profits)
plt.xlabel("Day")
plt.ylabel("Profit")
plt.title("RL Retailer Profit Over Time")
plt.show()
```

---

## 7. Visualizing Agent Communication

```python
import random
import matplotlib.pyplot as plt

G = nx.DiGraph()
G.add_edges_from([
    ("Customer","Retailer"),
    ("Retailer","Distributor"),
    ("Distributor","Factory"),
    ("Factory","Distributor"),
    ("Distributor","Retailer")
])

pos = nx.circular_layout(G)

plt.figure(figsize=(6,6))
nx.draw(G,pos,with_labels=True,node_size=2000,node_color="lightgreen",font_size=10)
plt.title("Message Passing in Supply Chain",fontsize=14)
plt.show()
```

---

## 🎯 Key Takeaways

- We built an **ontology** to formalize the supply chain domain.  
- With `messa`, agents exchanged **messages** (orders, deliveries).  
- The **retailer** used **reinforcement learning** to improve ordering decisions over time.  
- Visualizations clarified both ontology and communication structure.  

This notebook gives a compact but powerful demonstration of **multi-agent systems + RL in business supply chains**.  

---
